# Notebook 5 — Post-training inference (offline)

**What this notebook covers**

After `scripts/train.py` or Notebook 04 finishes, checkpoints live under `model/` and `artifacts/`. Here we reload them and match **the same logic** used in production inference (`app/main.py`): sparse feature row, TF–IDF slice for Naive Bayes, weighted soft voting.

**Conceptual “stages after training”**

1. **Load** vectorisers, scaler, classifiers, ensemble weights.
2. **Transform** raw prompt + two responses → one sparse row `X`.
3. **Infer** probabilities → predicted class + confidence.

**Canonical scripts**

| Role | Location |
|---|---|
| CLI test | `scripts/predict.py` |
| Shared feature+kernels | `app/main.py` → `build_feature_vector`, `/predict` |

**Optional spaCy NER parity:** Training with `--enable-spacy` ⇒ set **`ENABLE_SPACY_NER=1`** **before** the first `import app.main` in this kernel (then restart kernel if needed).

## Paths and imports

We add the project root so `import app.main` works without installing the repo as a package.

In [ ]:
import ast
import json
import sys
from pathlib import Path

import joblib
import numpy as np

ROOT = Path("..").resolve()
assert (ROOT / "app" / "main.py").exists(), f"Wrong ROOT: {ROOT}"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

MODEL_DIR = ROOT / "model"
ARTIFACT_DIR = ROOT / "artifacts"

print("ROOT:", ROOT)
print(
    "artifacts:", ARTIFACT_DIR.is_dir(),
    "| model:", MODEL_DIR.is_dir(),
)

## Load checkpoints into `app.main` globals

In [ ]:
import app.main as api

def load_checkpoints_into_api(
    model_dir: Path | None = None,
    artifact_dir: Path | None = None,
):
    md = Path(model_dir or MODEL_DIR)
    ad = Path(artifact_dir or ARTIFACT_DIR)

    api.tfidf_full = joblib.load(ad / "tfidf_full.pkl")
    api.tfidf_diff = joblib.load(ad / "tfidf_diff.pkl")
    api.scaler = joblib.load(ad / "scaler.pkl")
    api.model_sgd = joblib.load(md / "final_sgd.pkl")
    api.model_nb = joblib.load(md / "final_nb.pkl")
    api.model_lr = joblib.load(md / "final_logreg.pkl")
    api.ens_weights = np.load(md / "ensemble_weights.npy")
    with open(md / "model_metadata.json") as f:
        api.metadata = json.load(f)

    api.N_DENSE = int(
        getattr(api.scaler, "n_features_in_", len(api.scaler.scale_))
    )
    meta_n = api.metadata.get("n_dense_features")
    if meta_n is not None and int(meta_n) != api.N_DENSE:
        print(
            f"⚠ scaler dense={api.N_DENSE} vs metadata n_dense_features={meta_n}"
        )


load_checkpoints_into_api()

LABEL_MAP = {0: "Model A wins", 1: "Model B wins", 2: "Tie"}

print("N_DENSE =", api.N_DENSE, "| ensemble weights:", api.ens_weights)

## Ensemble helper (matches `/predict`)

In [ ]:
def ensemble_proba(api_mod, X_csr):
    n_tfidf = X_csr.shape[1] - api_mod.N_DENSE
    X_tfidf = X_csr[:, :n_tfidf]
    w_sgd, w_nb, w_lr = api_mod.ens_weights
    return (
        w_sgd * api_mod.model_sgd.predict_proba(X_csr)
        + w_nb * api_mod.model_nb.predict_proba(X_tfidf)
        + w_lr * api_mod.model_lr.predict_proba(X_csr)
    )

## Demo: one preference prediction

In [ ]:
prompt = "Explain gradient descent in two sentences."
response_a = (
    "Gradient descent minimises loss by stepping opposite an estimate of "
    "the gradient. It repeats until convergence."
)
response_b = "It is basically walking downhill."

X_vec, insights = api.build_feature_vector(prompt, response_a, response_b)
p_row = ensemble_proba(api, X_vec)[0]
label = int(np.argmax(p_row))

print("Shape:", X_vec.shape)
print("Sample insights:", dict(list(insights.items())[:4]))
print()
print("Prediction:", LABEL_MAP[label])
print("Confidence (max prob):", float(p_row[label]))
for i in range(3):
    print(f"  {LABEL_MAP[i]}:", float(p_row[i]))

## Tiny batch from `train.csv`

`parse_field` mirrors Notebook 02 / `scripts/train.py`, so nested JSON chat cells become plain text before inference.

In [ ]:
import pandas as pd


def parse_field(val):
    if pd.isna(val):
        return ""
    v = str(val).strip()
    if v.startswith("["):
        try:
            return " ".join(str(p) for p in json.loads(v))
        except Exception:
            try:
                return " ".join(str(p) for p in ast.literal_eval(v))
            except Exception:
                return v
    return v


csv_path = ROOT / "data" / "train.csv"
if not csv_path.exists():
    raise FileNotFoundError(f"Expected {csv_path} (provide data or skip this cell)")

df = pd.read_csv(csv_path, nrows=80)

rows_out = []
for i, row in df.head(5).iterrows():
    p_txt = parse_field(row["prompt"])
    a_txt = parse_field(row["response_a"])
    b_txt = parse_field(row["response_b"])
    xv, _ = api.build_feature_vector(p_txt, a_txt, b_txt)
    probs = ensemble_proba(api, xv)[0]

    rows_out.append(
        {
            "csv_row": i,
            "prob_A": probs[0],
            "prob_B": probs[1],
            "prob_Tie": probs[2],
        }
    )

pd.DataFrame(rows_out)